In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-26 16:42:51--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  --.-KB/s    in 0.1s    

2026-11-26 16:42:51 (192 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
books = pd.read_csv(books_filename, sep=';', encoding='latin-1',
                    on_bad_lines='skip', low_memory=False)

ratings = pd.read_csv(ratings_filename, sep=';', encoding='latin-1',
                      on_bad_lines='skip', low_memory=False)

print("Books shape:", books.shape)
print("Ratings shape:", ratings.shape)

display(books.head())
display(ratings.head())

Books shape: (271360, 8)
Ratings shape: (1149780, 3)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [4]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [5]:
# add your code here - consider creating a new cell for each section of code

In [8]:

# Calculate user counts from the original ratings
user_counts = df_ratings['user'].value_counts()

# Calculate book counts from the original ratings
book_counts = df_ratings['isbn'].value_counts()

# Select users with at least 200 ratings
users_to_keep = user_counts[user_counts >= 200].index

# Select books with at least 100 ratings
books_to_keep = book_counts[book_counts >= 100].index

# Filter both independently using original counts
filtered_ratings = df_ratings[
    df_ratings['user'].isin(users_to_keep) &
    df_ratings['isbn'].isin(books_to_keep)
]

# Merge ratings with book information
df = pd.merge(filtered_ratings, df_books, on='isbn')

# Remove duplicate user-book combinations
df = df.drop_duplicates(['user', 'title'])

print("Filtered dataset shape:", df.shape)
print("Unique users:", df['user'].nunique())
print("Unique books:", df['title'].nunique())


Filtered dataset shape: (49136, 5)
Unique users: 888
Unique books: 673


In [9]:

# Create user-book rating matrix
book_rating_pivot = df.pivot(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

print("Rating matrix shape:", book_rating_pivot.shape)

# Convert matrix into sparse format
book_rating_matrix = csr_matrix(book_rating_pivot.values)

# Create and train KNN model
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(book_rating_matrix)

print("KNN model trained successfully!")


Rating matrix shape: (673, 888)
KNN model trained successfully!


In [14]:

def get_recommends(book=""):

    recommended_books = [book]

    # Get the six nearest neighbours
    distances, indices = model.kneighbors(
        book_rating_pivot.loc[book].values.reshape(1, -1),
        n_neighbors=6
    )

    recommendations = []

    # Skip the first two neighbours
    for i in range(2, len(distances.flatten())):
        recommendations.append([
            book_rating_pivot.index[indices.flatten()[i]],
            float(distances.flatten()[i])
        ])

    # Reverse the recommendation order
    recommended_books.append(recommendations[::-1])

    return recommended_books


In [15]:
recommendations = get_recommends(
    "The Queen of the Damned (Vampire Chronicles (Paperback))"
)

print(recommendations)

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['Catch 22', 0.793983519077301], ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448656558990479], ['Interview with the Vampire', 0.7345068454742432], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.5376338362693787]]]


In [16]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074]]]
You passed the challenge! 🎉🎉🎉🎉🎉
